# Imports and Setup

In [1]:
# --- Core PyTorch Libraries ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# --- Data Handling and Image Processing ---
import numpy as np
import rasterio
import glob
import os

# --- Visualization & Progress ---
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# --- (Optional) Augmentations ---
# import albumentations as A
# from albumentations.pytorch import ToTensorV2

print("All libraries imported successfully!")

All libraries imported successfully!


# The Dataset Class

We'll use the same robust GlacialLakeDataset class we finalized earlier. It already handles the multi-modal input (stacking 8 channels) which is a key part of your proposed solution. There's no need to change it

In [2]:
class GlacialLakeDataset(Dataset):
    """
    Dataset with correct normalization for all 8 channels (Optical, Radar, and DEM-derived).
    """
    def __init__(self, image_dir, mask_dir, augmentations=None, target_size=256):
        self.image_dir, self.mask_dir, self.augmentations, self.target_size = image_dir, mask_dir, augmentations, target_size
        self.pairs = []
        image_files = sorted(glob.glob(os.path.join(image_dir, "*.tif")))
        for img_path in image_files:
            filename = os.path.basename(img_path)
            if "input_stack_chunk" in filename:
                parts = filename.split("_")
                if len(parts) >= 5:
                    scene_id, chunk_id = parts[1], parts[-1].replace(".tif", "")
                    key = f"LISS3_{scene_id}_lake_mask_chunk_{chunk_id}.tif"
                    mask_path = os.path.join(self.mask_dir, key)
                    if os.path.exists(mask_path): self.pairs.append((img_path, mask_path))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        with rasterio.open(img_path) as src: image = src.read().astype(np.float32)
        with rasterio.open(mask_path) as src: mask = src.read(1).astype(np.float32)
        
        image[:4, :, :] /= 1023.0 
        DEM_MAX, SLOPE_MAX, ASPECT_MAX = 4000.0, 90.0, 360.0
        image[4, :, :] /= DEM_MAX 
        image[5, :, :] /= SLOPE_MAX
        image[6, :, :] /= ASPECT_MAX
        image[7, :, :] = (image[7, :, :] + 1.0) / 2.0
        image = np.clip(image, 0.0, 1.0)

        num_channels, h, w = image.shape
        padded_image = np.zeros((num_channels, self.target_size, self.target_size), dtype=np.float32)
        padded_mask = np.zeros((self.target_size, self.target_size), dtype=np.float32)
        padded_image[:, :h, :w], padded_mask[:h, :w] = image, mask
        
        if self.augmentations:
            augmented = self.augmentations(image=padded_image.transpose(1, 2, 0), mask=padded_mask)
            padded_image, padded_mask = augmented['image'].transpose(2, 0, 1), augmented['mask']
            
        return {"image": torch.from_numpy(padded_image), "mask": torch.from_numpy(padded_mask).unsqueeze(0)}

print("Cell 2/6: GlacialLakeDataset class defined.")

Cell 2/6: GlacialLakeDataset class defined.


# The Hybrid Model Building Blocks

Now, let's define the components for the new model. We'll start with the standard blocks, then add the 

Attention Gate which is the core of the Attention U-Net part of your hybrid design

In [5]:
# --- Standard U-Net Blocks ---
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels: mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, padding=1, bias=False), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))
    def forward(self, x): return self.maxpool_conv(x)

# --- GLNet-Style Encoder Branches ---
class GlobalBranch(nn.Module):
    def __init__(self, in_channels, out_channels=32):
        super(GlobalBranch, self).__init__()
        self.downsample = nn.AvgPool2d(4, stride=4)
        self.convs = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
        self.upsample = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)
    def forward(self, x): return self.upsample(self.convs(self.downsample(x)))

class LocalBranch(nn.Module):
    def __init__(self, in_channels, out_channels=32):
        super(LocalBranch, self).__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.convs(x)

# --- Attention U-Net Decoder Blocks ---
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x): return x * self.psi(self.relu(self.W_g(g) + self.W_x(x)))

# In Cell 3, replace the old Up_Attention class with this one:

# In Cell 3, replace the old Up_Attention class with this one:

class Up_Attention(nn.Module):
    """
    Upscaling then double conv, with attention gate and dropout [CORRECTED]
    """
    def __init__(self, ch_g, ch_x, ch_out, dropout_p=0.5):
        # ch_g: channels for the gating signal (from lower layer)
        # ch_x: channels for the skip connection
        # ch_out: output channels
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.att = AttentionGate(F_g=ch_g, F_l=ch_x, F_int=(ch_g + ch_x) // 4)
        self.conv = DoubleConv(in_channels=ch_g + ch_x, out_channels=ch_out)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x1, x2):
        # x1 is from lower layer (gating signal), x2 is the skip connection
        x1 = self.up(x1)
        x2_att = self.att(g=x1, x=x2)
        
        # Handle potential size mismatches
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        
        # Concatenate and apply convolutions/dropout
        x = torch.cat([x2_att, x1], dim=1)
        return self.conv(self.dropout(x))

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x): return self.conv(x)

print("Cell 3/6: All model building blocks defined.")

Cell 3/6: All model building blocks defined.


# The Full Hybrid Model Architecture


In [6]:
# In Cell 4, replace the old GlacialLake_HybridNet class with this one:

class GlacialLake_HybridNet(nn.Module):
    """
    The full Hybrid model, with a GLNet parallel encoder, an attention-based
    decoder, and Monte Carlo Dropout layers. [CORRECTED]
    """
    def __init__(self, n_channels, n_classes, dropout_p=0.5):
        super(GlacialLake_HybridNet, self).__init__()
        self.local_branch = LocalBranch(n_channels, out_channels=32)
        self.global_branch = GlobalBranch(n_channels, out_channels=32)
        
        # --- Downward Path ---
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        
        # --- Attention-based Decoder Path [FIXED INITIALIZATION] ---
        # Signature: Up_Attention(ch_g, ch_x, ch_out, dropout_p)
        self.up1 = Up_Attention(1024, 512, 512, dropout_p)
        self.up2 = Up_Attention(512, 256, 256, dropout_p)
        self.up3 = Up_Attention(256, 128, 128, dropout_p)
        self.up4 = Up_Attention(128, 64, 64, dropout_p)
        
        # --- Output Layer ---
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        # GLNet Encoder
        x1 = torch.cat([self.local_branch(x), self.global_branch(x)], dim=1)
        
        # Downward Path
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        # Attention Decoder Path
        x_up = self.up1(x5, x4)
        x_up = self.up2(x_up, x3)
        x_up = self.up3(x_up, x2)
        x_up = self.up4(x_up, x1)
        
        return self.outc(x_up)

print("Cell 4/6: Corrected GlacialLake_HybridNet class defined.")

Cell 4/6: Corrected GlacialLake_HybridNet class defined.


# Boundary-Aware Loss Function

In [7]:
class BoundaryLoss(nn.Module):
    """
    Combines Dice + BCE with a loss focused on the boundaries to produce
    sharper, more accurate lake outlines.
    """
    def __init__(self, dice_bce_weight=0.5, boundary_weight=0.5, smooth=1e-6):
        super(BoundaryLoss, self).__init__()
        self.dice_bce_weight, self.boundary_weight, self.smooth = dice_bce_weight, boundary_weight, smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        probs, probs_flat, targets_flat = torch.sigmoid(logits), logits.view(-1), targets.view(-1)
        intersection = (probs.view(-1) * targets_flat).sum()
        dice_loss = 1 - (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        standard_loss = (self.dice_bce_weight * self.bce(logits, targets)) + ((1-self.dice_bce_weight) * dice_loss)
        
        min_pool = F.max_pool2d(1 - targets, 3, 1, 1)
        boundary_targets = (targets - (1 - min_pool)).clamp(0, 1)
        boundary_bce = F.binary_cross_entropy_with_logits(logits, boundary_targets, weight=boundary_targets)
        
        return standard_loss + (self.boundary_weight * boundary_bce)

print("Cell 5/6: BoundaryLoss class defined.")

Cell 5/6: BoundaryLoss class defined.


# Configuration and Initialization

This final cell sets up the main configuration variables and creates instances of the dataset, model, loss function, and optimizer, making everything ready for training.

In [8]:
# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_IMG_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\input_stack\train"
TRAIN_MSK_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\lake_mask\train"
VAL_IMG_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\input_stack\validate"
VAL_MSK_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\lake_mask\validate"
LEARNING_RATE = 1e-4
BATCH_SIZE = 4 # Adjust based on your GPU memory

# --- Initialization ---
train_dataset = GlacialLakeDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dataset = GlacialLakeDataset(VAL_IMG_DIR, VAL_MSK_DIR)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = GlacialLake_HybridNet(n_channels=8, n_classes=1).to(DEVICE)
loss_fn = BoundaryLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler()

print(f"Cell 6/6: Configuration and initialization complete. Using device: {DEVICE}")
print(f"Model, Dataloaders, Loss Function, and Optimizer are ready for training.")

Cell 6/6: Configuration and initialization complete. Using device: cuda
Model, Dataloaders, Loss Function, and Optimizer are ready for training.


#  The Training Function

This cell contains a function that handles all the logic for a single training epoch, including the forward pass, loss calculation, backpropagation, and optimizer step. It uses mixed-precision training for better performance.

In [9]:
def train_one_epoch(loader, model, optimizer, loss_fn, scaler, device="cuda"):
    """
    Performs one full epoch of training.
    """
    loop = tqdm(loader, leave=True)
    loop.set_description(f"Training")
    
    total_loss = 0.0
    
    for batch in loop:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        
        # Forward pass with Automatic Mixed Precision
        with torch.amp.autocast(device_type=device):
            predictions = model(images)
            loss = loss_fn(predictions, masks)
        
        # Backward pass
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    return total_loss / len(loader)

print("Cell 7/9: Training function defined.")

Cell 7/9: Training function defined.


# The Validation Function

This function evaluates the model on the validation dataset. It calculates both the validation loss and the Dice Score, which is a better measure of segmentation quality.

Note: For standard validation, we set model.eval() to disable dropout. We will address the special case for Monte Carlo uncertainty later.



In [10]:
def validate_model(loader, model, loss_fn, device="cuda"):
    """
    Evaluates the model on the validation dataset.
    Calculates validation loss and Dice score.
    """
    dice_score = 0
    val_loss = 0
    
    # Set model to evaluation mode
    model.eval()
    
    loop = tqdm(loader, leave=True)
    loop.set_description(f"Validation")
    
    # Disable gradient calculations for validation
    with torch.no_grad():
        for batch in loop:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)
            
            # Forward pass
            predictions = model(images)
            
            # Calculate validation loss
            val_loss += loss_fn(predictions, masks).item()
            
            # Calculate Dice Score
            preds_binary = torch.sigmoid(predictions) > 0.5
            dice_score += (2. * (preds_binary * masks).sum()) / ((preds_binary + masks).sum() + 1e-8)
            
            loop.set_postfix(dice=f"{(dice_score / len(loader)):.4f}")

    # Set model back to training mode
    model.train()
    
    avg_val_loss = val_loss / len(loader)
    avg_dice_score = dice_score / len(loader)
    
    return avg_val_loss, avg_dice_score

print("Cell 8/9: Validation function defined.")

Cell 8/9: Validation function defined.


# The Main Training Loop

In [11]:
# --- Training Configuration ---
NUM_EPOCHS = 50 # You can increase this for a full training run
MODEL_SAVE_PATH = "hybrid_model_best.pth"

# --- Main Loop ---
best_dice_score = -1.0

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    
    # Run one epoch of training
    train_loss = train_one_epoch(train_loader, model, optimizer, loss_fn, scaler, device=DEVICE)
    
    # Run validation
    current_val_loss, current_dice_score = validate_model(val_loader, model, loss_fn, device=DEVICE)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {current_val_loss:.4f} | Val Dice Score: {current_dice_score:.4f}")
    
    # Save the model if it has the best validation Dice score so far
    if current_dice_score > best_dice_score:
        best_dice_score = current_dice_score
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"🎉 New best model saved with Dice Score: {best_dice_score:.4f}")

print("\n✅ Training complete!")
print(f"Best model saved to {MODEL_SAVE_PATH} with a Dice score of {best_dice_score:.4f}")


--- Epoch 1/50 ---


  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/56 [00:00<?, ?it/s]

Train Loss: 0.6339 | Val Loss: 0.5848 | Val Dice Score: 0.0000
🎉 New best model saved with Dice Score: 0.0000

--- Epoch 2/50 ---


  0%|          | 0/445 [00:00<?, ?it/s]

...

  0%|          | 0/56 [00:00<?, ?it/s]

Train Loss: 0.3405 | Val Loss: 0.3722 | Val Dice Score: 0.3074

--- Epoch 50/50 ---


  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/56 [00:00<?, ?it/s]

Train Loss: 0.3442 | Val Loss: 0.3999 | Val Dice Score: 0.2534

✅ Training complete!
Best model saved to hybrid_model_best.pth with a Dice score of 0.3153
